# 1. Libraries

## 1.1 Own Modules

In [142]:
from modules.pdf_parser import extract_text_from_static_pdf
from modules.text_preprocessor import preprocess_text
from modules.qa_engine import generate_answer
from config.settings import TOGETHER_API_KEY, MODEL_NAME_1, MODEL_NAME_2, MODEL_NAME_3, MODEL_NAME_4

## 1.2 Python modules

In [143]:
# Streamlit
import streamlit as st

# Chunking
import fitz

# Excel
import openpyxl

# Testing
import pandas as pd
from typing import Union, IO
import requests
import time
import json

# Logging
import logging

# Variation Score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Cleaning expected keywords
import ast

# 2. Setup

## 2.1 File Name

In [144]:
file_path = "./tests/sample_docs/Sanjay_Prabhu_Kunjibettu_Resume.pdf"

## 2.2 Cosine Score setup

In [145]:
# Load once globally
embedder = SentenceTransformer('all-MiniLM-L6-v2')

[06/Jun/2025 23:19:49] INFO - Use pytorch device_name: cpu
[06/Jun/2025 23:19:49] INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


## 2.3 Models

In [146]:
models = [MODEL_NAME_1, MODEL_NAME_2, MODEL_NAME_3, MODEL_NAME_4]

## 2.4 Test Suite Dataframe

In [147]:
df_test_suite = pd.read_excel("./tests/test_suite.xlsx", engine="openpyxl")

# 3. Functions

## Cleaning Keywords

In [148]:
def clean_expected_keywords_list(raw_keywords):
    cleaned = []
    for kw in raw_keywords:
        if isinstance(kw, tuple):
            # Take first element if tuple
            kw = kw[0]
        # Convert to string just in case, and strip whitespace
        cleaned.append(str(kw).strip())
    return cleaned

## 3.1 Cosine Similarity

In [149]:
def semantic_similarity(text1: str, text2: str) -> float:
    embeddings = embedder.encode([text1, text2])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(similarity, 4)

## 3.2 Metadata Testing: Chunking, Token Size

In [150]:
def prepare_context(file_path, num_chunks=15):
    start = time.time()
    text = extract_text_from_static_pdf(file_path)
    chunks = preprocess_text(text)
    end = time.time()

    context = "\n".join(chunks[:num_chunks])
    metadata = {
        "Chunking Time (s)": round(end - start, 2),
        "Total Chunks": len(chunks),
        "Total Characters": len(text)
    }
    return context, metadata

## 3.3 Generate/Score Response

In [151]:
def generate_and_score_response(prompt, model, expected_keywords, reference_response=None, context=None):
    start = time.time()
    response = generate_answer(context, prompt, model)
    end = time.time()

    response_time = round(end - start, 2)
    matched_keywords = [kw for kw in expected_keywords if kw.lower() in response.lower()]
    consistency_score = round(len(matched_keywords) / len(expected_keywords), 2) if expected_keywords else 0.0
    similarity_score = round(semantic_similarity(response, reference_response), 2) if reference_response else 1.0

    return {
        "Prompt": prompt,
        "Response": response,
        "Response Time (s)": response_time,
        "Consistency Score": consistency_score,
        "Similarity to Baseline": similarity_score
    }

## 3.4 Batch Processing

In [152]:
def process_batch(batch_df, model, iteration, results_df, context, delay=10):
    baseline_row = batch_df.iloc[0]
    variation_rows = batch_df.iloc[1:3]

    prompt_baseline = baseline_row["question"]
    expected_keywords = baseline_row["expected_keywords"]

    if isinstance(expected_keywords, str):
        # maybe parse if string representation of list
        try:
            expected_keywords = ast.literal_eval(expected_keywords)
        except Exception:
            expected_keywords = [expected_keywords]
    elif not isinstance(expected_keywords, list):
        expected_keywords = [expected_keywords]

    expected_keywords = clean_expected_keywords_list(expected_keywords)

    # Baseline
    result = generate_and_score_response(prompt_baseline, model, expected_keywords, context=context)
    result.update({
        "Iteration": iteration,
        "Model": model,
        "Prompt Type": "Baseline"
    })
    results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
    time.sleep(delay)

    # Variations
    reference_response = result["Response"]
    for _, row in variation_rows.iterrows():
        variation_result = generate_and_score_response(row["question"], model, expected_keywords,
                                                       reference_response, context=context)
        variation_result.update({
            "Iteration": iteration,
            "Model": model,
            "Prompt Type": "Variation"
        })
        results_df = pd.concat([results_df, pd.DataFrame([variation_result])], ignore_index=True)
        time.sleep(delay)

    return results_df

## 3.5 Running Test Suite

In [153]:
def run_test_suite(df_test_suite, models, file_path, iterations=5, delay=10):
    df_results = pd.DataFrame()
    full_context, metadata = prepare_context(file_path)

    logging.info(f"Chunking Metadata: {metadata}")

    for iteration in range(1, iterations + 1):
        logging.info(f"--- Iteration {iteration} ---")
        iter_start = time.time()

        for i in range(0, len(df_test_suite), 3):
            batch_df = df_test_suite.iloc[i:i + 3]
            for model in models:
                df_results = process_batch(batch_df, model, iteration, df_results, full_context, delay)

        iter_time = round(time.time() - iter_start, 2)
        logging.info(f"Iteration {iteration} completed in {iter_time} seconds.")

    return df_results, metadata

# 4. Test

In [ ]:
df_results, meta = run_test_suite(df_test_suite, models, file_path, iterations=5, delay=10)

[06/Jun/2025 23:20:24] INFO - Chunking Metadata: {'Chunking Time (s)': 0.03, 'Total Chunks': 1, 'Total Characters': 4272}
[06/Jun/2025 23:20:24] INFO - --- Iteration 1 ---


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.55it/s]


In [ ]:
df_results.to_csv("./tests/results/log.csv", index=False)

,Iteration,Model,Prompt Type,Prompt,Response,Response Time (s),Consistency Score,Similarity to Baseline
0,1,meta-llama/Llama-3.3-70B-Instruct-Turbo-Free,Baseline,What is the full name of the person in this do...,Sanjay Prabhu Kunjibettu.,3.09,0.0,1.00
1,1,meta-llama/Llama-3.3-70B-Instruct-Turbo-Free,Variation,Please tell me the candidate's name.,Sanjay Prabhu Kunjibettu.,1.16,0.0,1.00
2,1,meta-llama/Llama-3.3-70B-Instruct-Turbo-Free,Variation,Can you identify the individual in the document?,The individual in the document is Sanjay Prabh...,1.89,0.0,0.82
3,1,deepseek-ai/DeepSeek-R1-Distill-Llama-70B-free,Baseline,What is the full name of the person in this do...,Sanjay Prabhu Kunjibettu\n</think><think>\nAlr...,11.05,0.0,1.00
4,1,deepseek-ai/DeepSeek-R1-Distill-Llama-70B-free,Variation,Please tell me the candidate's name.,The candidate's name is Sanjay Prabhu Kunjibet...,7.64,0.0,0.84
...,...,...,...,...,...,...,...,...
57,1,lgai/exaone-deep-32b,Baseline,Mention any awards or recognitions received by...,The candidate has received the following award...,5.64,0.0,1.00
58,1,lgai/exaone-deep-32b,Variation,"Has the individual received any honors, awards...",The individual has received several honors and...,5.49,0.0,0.90
59,1,lgai/exaone-deep-32b,Variation,List any formal recognitions or accolades earn...,"The answer should be listed in bullet points, ...",3.75,0.0,0.87
60,2,meta-llama/Llama-3.3-70B-Instruct-Turbo-Free,Baseline,What is the full name of the person in this do...,Sanjay Prabhu Kunjibettu.,1.01,0.0,1.00


In [ ]:
logging.info("Test results saved to ./tests/results/log.csv")